In [0]:
%sql
create schema if not exists retail_sales_dashboard;

In [0]:
%sql
use schema retail_sales_dashboard;

In [0]:
%sql
select current_catalog();

current_catalog()
sathya_databricks123


In [0]:
%sql
select current_schema();

current_schema()
retail_sales_dashboard


In [0]:
sales_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sathya_databricks123/default/filestore/sales.csv")

products_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("/Volumes/sathya_databricks123/default/filestore/products.csv")

In [0]:
display(sales_df)

sale_id,product_id,store_id,employee_id,quantity,sale_date
1,P101,S101,201,2,2026-06-01
2,P102,S102,202,3,2026-06-01
3,P103,S103,203,5,2026-06-01
4,P104,S104,204,4,2026-06-01
5,P105,S105,205,2,2026-06-01
6,P106,S101,206,3,2026-06-01
7,P107,S102,207,8,2026-06-01
8,P108,S103,208,10,2026-06-01
9,P109,S104,209,2,2026-06-01
10,P110,S105,210,4,2026-06-01


In [0]:
display(products_df)

product_id,product_name,category,price,cost
P101,Laptop,Electronics,65000.0,55000.0
P102,Smartphone,Electronics,30000.0,25000.0
P103,Headphones,Electronics,2500.0,1800.0
P104,Smartwatch,Electronics,5000.0,3500.0
P105,Tablet,Electronics,20000.0,16000.0
P106,Printer,Accessories,12000.0,9000.0
P107,Keyboard,Accessories,1500.0,1000.0
P108,Mouse,Accessories,800.0,500.0
P109,Monitor,Electronics,15000.0,12000.0
P110,Webcam,Accessories,2500.0,1800.0


In [0]:
sales_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)



In [0]:
products_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)



In [0]:
print("Sales Records :", sales_df.count())

Sales Records : 20


In [0]:
print("Product Records :", products_df.count())

Product Records : 10


In [0]:
sales_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("sathya_databricks123.retail_sales_dashboard.bronze_sales")

In [0]:
products_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("sathya_databricks123.retail_sales_dashboard.bronze_products")

In [0]:
%sql
select * from sathya_databricks123.retail_sales_dashboard.bronze_sales;

sale_id,product_id,store_id,employee_id,quantity,sale_date
1,P101,S101,201,2,2026-06-01
2,P102,S102,202,3,2026-06-01
3,P103,S103,203,5,2026-06-01
4,P104,S104,204,4,2026-06-01
5,P105,S105,205,2,2026-06-01
6,P106,S101,206,3,2026-06-01
7,P107,S102,207,8,2026-06-01
8,P108,S103,208,10,2026-06-01
9,P109,S104,209,2,2026-06-01
10,P110,S105,210,4,2026-06-01


In [0]:
%sql
select * from sathya_databricks123.retail_sales_dashboard.bronze_products;

product_id,product_name,category,price,cost
P101,Laptop,Electronics,65000.0,55000.0
P102,Smartphone,Electronics,30000.0,25000.0
P103,Headphones,Electronics,2500.0,1800.0
P104,Smartwatch,Electronics,5000.0,3500.0
P105,Tablet,Electronics,20000.0,16000.0
P106,Printer,Accessories,12000.0,9000.0
P107,Keyboard,Accessories,1500.0,1000.0
P108,Mouse,Accessories,800.0,500.0
P109,Monitor,Electronics,15000.0,12000.0
P110,Webcam,Accessories,2500.0,1800.0


In [0]:
bronze_sales = spark.table(
    "sathya_databricks123.retail_sales_dashboard.bronze_sales"
)

bronze_products = spark.table(
    "sathya_databricks123.retail_sales_dashboard.bronze_products"
)

display(bronze_sales)
display(bronze_products)

sale_id,product_id,store_id,employee_id,quantity,sale_date
1,P101,S101,201,2,2026-06-01
2,P102,S102,202,3,2026-06-01
3,P103,S103,203,5,2026-06-01
4,P104,S104,204,4,2026-06-01
5,P105,S105,205,2,2026-06-01
6,P106,S101,206,3,2026-06-01
7,P107,S102,207,8,2026-06-01
8,P108,S103,208,10,2026-06-01
9,P109,S104,209,2,2026-06-01
10,P110,S105,210,4,2026-06-01


product_id,product_name,category,price,cost
P101,Laptop,Electronics,65000.0,55000.0
P102,Smartphone,Electronics,30000.0,25000.0
P103,Headphones,Electronics,2500.0,1800.0
P104,Smartwatch,Electronics,5000.0,3500.0
P105,Tablet,Electronics,20000.0,16000.0
P106,Printer,Accessories,12000.0,9000.0
P107,Keyboard,Accessories,1500.0,1000.0
P108,Mouse,Accessories,800.0,500.0
P109,Monitor,Electronics,15000.0,12000.0
P110,Webcam,Accessories,2500.0,1800.0


In [0]:
from pyspark.sql.functions import col, sum

display(
    bronze_sales.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in bronze_sales.columns
    ])
)

sale_id,product_id,store_id,employee_id,quantity,sale_date
0,0,0,0,0,0


In [0]:
display(
    bronze_products.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in bronze_products.columns
    ])
)

product_id,product_name,category,price,cost
0,0,0,0,0


In [0]:
silver_sales = bronze_sales.dropDuplicates()

silver_products = bronze_products.dropDuplicates()

In [0]:
print(bronze_sales.count())

print(silver_sales.count())

20
20


In [0]:
silver_sales_products = silver_sales.join(
    silver_products,
    "product_id",
    "inner"
)

display(silver_sales_products)

product_id,sale_id,store_id,employee_id,quantity,sale_date,product_name,category,price,cost
P101,1,S101,201,2,2026-06-01,Laptop,Electronics,65000.0,55000.0
P107,7,S102,207,8,2026-06-01,Keyboard,Accessories,1500.0,1000.0
P109,9,S104,209,2,2026-06-01,Monitor,Electronics,15000.0,12000.0
P108,8,S103,208,10,2026-06-01,Mouse,Accessories,800.0,500.0
P110,10,S105,210,4,2026-06-01,Webcam,Accessories,2500.0,1800.0
P104,4,S104,204,4,2026-06-01,Smartwatch,Electronics,5000.0,3500.0
P110,20,S105,210,5,2026-06-02,Webcam,Accessories,2500.0,1800.0
P102,2,S102,202,3,2026-06-01,Smartphone,Electronics,30000.0,25000.0
P105,5,S105,205,2,2026-06-01,Tablet,Electronics,20000.0,16000.0
P101,11,S101,201,1,2026-06-02,Laptop,Electronics,65000.0,55000.0


In [0]:
from pyspark.sql.functions import col

silver_sales_products = silver_sales_products.withColumn(
    "revenue",
    col("quantity") * col("price")
)

display(silver_sales_products)

product_id,sale_id,store_id,employee_id,quantity,sale_date,product_name,category,price,cost,revenue
P101,1,S101,201,2,2026-06-01,Laptop,Electronics,65000.0,55000.0,130000.0
P107,7,S102,207,8,2026-06-01,Keyboard,Accessories,1500.0,1000.0,12000.0
P109,9,S104,209,2,2026-06-01,Monitor,Electronics,15000.0,12000.0,30000.0
P108,8,S103,208,10,2026-06-01,Mouse,Accessories,800.0,500.0,8000.0
P110,10,S105,210,4,2026-06-01,Webcam,Accessories,2500.0,1800.0,10000.0
P104,4,S104,204,4,2026-06-01,Smartwatch,Electronics,5000.0,3500.0,20000.0
P110,20,S105,210,5,2026-06-02,Webcam,Accessories,2500.0,1800.0,12500.0
P102,2,S102,202,3,2026-06-01,Smartphone,Electronics,30000.0,25000.0,90000.0
P105,5,S105,205,2,2026-06-01,Tablet,Electronics,20000.0,16000.0,40000.0
P101,11,S101,201,1,2026-06-02,Laptop,Electronics,65000.0,55000.0,65000.0


In [0]:
silver_sales_products = silver_sales_products.withColumn(
    "cost_amount",
    col("quantity") * col("cost")
)

In [0]:
silver_sales_products = silver_sales_products.withColumn(
    "profit",
    col("revenue") - col("cost_amount")
)

display(silver_sales_products)

product_id,sale_id,store_id,employee_id,quantity,sale_date,product_name,category,price,cost,revenue,cost_amount,profit
P101,1,S101,201,2,2026-06-01,Laptop,Electronics,65000.0,55000.0,130000.0,110000.0,20000.0
P107,7,S102,207,8,2026-06-01,Keyboard,Accessories,1500.0,1000.0,12000.0,8000.0,4000.0
P109,9,S104,209,2,2026-06-01,Monitor,Electronics,15000.0,12000.0,30000.0,24000.0,6000.0
P108,8,S103,208,10,2026-06-01,Mouse,Accessories,800.0,500.0,8000.0,5000.0,3000.0
P110,10,S105,210,4,2026-06-01,Webcam,Accessories,2500.0,1800.0,10000.0,7200.0,2800.0
P104,4,S104,204,4,2026-06-01,Smartwatch,Electronics,5000.0,3500.0,20000.0,14000.0,6000.0
P110,20,S105,210,5,2026-06-02,Webcam,Accessories,2500.0,1800.0,12500.0,9000.0,3500.0
P102,2,S102,202,3,2026-06-01,Smartphone,Electronics,30000.0,25000.0,90000.0,75000.0,15000.0
P105,5,S105,205,2,2026-06-01,Tablet,Electronics,20000.0,16000.0,40000.0,32000.0,8000.0
P101,11,S101,201,1,2026-06-02,Laptop,Electronics,65000.0,55000.0,65000.0,55000.0,10000.0


In [0]:
silver_sales_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.retail_sales_dashboard.silver_sales_products"
)

In [0]:
%sql
select *
from sathya_databricks123.retail_sales_dashboard.silver_sales_products;

product_id,sale_id,store_id,employee_id,quantity,sale_date,product_name,category,price,cost,revenue,cost_amount,profit
P101,1,S101,201,2,2026-06-01,Laptop,Electronics,65000.0,55000.0,130000.0,110000.0,20000.0
P107,7,S102,207,8,2026-06-01,Keyboard,Accessories,1500.0,1000.0,12000.0,8000.0,4000.0
P109,9,S104,209,2,2026-06-01,Monitor,Electronics,15000.0,12000.0,30000.0,24000.0,6000.0
P108,8,S103,208,10,2026-06-01,Mouse,Accessories,800.0,500.0,8000.0,5000.0,3000.0
P110,10,S105,210,4,2026-06-01,Webcam,Accessories,2500.0,1800.0,10000.0,7200.0,2800.0
P104,4,S104,204,4,2026-06-01,Smartwatch,Electronics,5000.0,3500.0,20000.0,14000.0,6000.0
P110,20,S105,210,5,2026-06-02,Webcam,Accessories,2500.0,1800.0,12500.0,9000.0,3500.0
P102,2,S102,202,3,2026-06-01,Smartphone,Electronics,30000.0,25000.0,90000.0,75000.0,15000.0
P105,5,S105,205,2,2026-06-01,Tablet,Electronics,20000.0,16000.0,40000.0,32000.0,8000.0
P101,11,S101,201,1,2026-06-02,Laptop,Electronics,65000.0,55000.0,65000.0,55000.0,10000.0


In [0]:
silver_df = spark.table(
    "sathya_databricks123.retail_sales_dashboard.silver_sales_products"
)

display(silver_df)

product_id,sale_id,store_id,employee_id,quantity,sale_date,product_name,category,price,cost,revenue,cost_amount,profit
P101,1,S101,201,2,2026-06-01,Laptop,Electronics,65000.0,55000.0,130000.0,110000.0,20000.0
P107,7,S102,207,8,2026-06-01,Keyboard,Accessories,1500.0,1000.0,12000.0,8000.0,4000.0
P109,9,S104,209,2,2026-06-01,Monitor,Electronics,15000.0,12000.0,30000.0,24000.0,6000.0
P108,8,S103,208,10,2026-06-01,Mouse,Accessories,800.0,500.0,8000.0,5000.0,3000.0
P110,10,S105,210,4,2026-06-01,Webcam,Accessories,2500.0,1800.0,10000.0,7200.0,2800.0
P104,4,S104,204,4,2026-06-01,Smartwatch,Electronics,5000.0,3500.0,20000.0,14000.0,6000.0
P110,20,S105,210,5,2026-06-02,Webcam,Accessories,2500.0,1800.0,12500.0,9000.0,3500.0
P102,2,S102,202,3,2026-06-01,Smartphone,Electronics,30000.0,25000.0,90000.0,75000.0,15000.0
P105,5,S105,205,2,2026-06-01,Tablet,Electronics,20000.0,16000.0,40000.0,32000.0,8000.0
P101,11,S101,201,1,2026-06-02,Laptop,Electronics,65000.0,55000.0,65000.0,55000.0,10000.0


In [0]:
from pyspark.sql.functions import sum

gold_product_summary = silver_df.groupBy(
    "product_id",
    "product_name"
).agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue"),
    sum("profit").alias("total_profit")
)

display(gold_product_summary)

product_id,product_name,total_quantity,total_revenue,total_profit
P101,Laptop,3,195000.0,30000.0
P107,Keyboard,15,22500.0,7500.0
P109,Monitor,3,45000.0,9000.0
P108,Mouse,19,15200.0,5700.0
P110,Webcam,9,22500.0,6300.0
P104,Smartwatch,7,35000.0,10500.0
P102,Smartphone,5,150000.0,25000.0
P105,Tablet,3,60000.0,12000.0
P103,Headphones,11,27500.0,7700.0
P106,Printer,5,60000.0,15000.0


In [0]:
gold_product_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.retail_sales_dashboard.gold_product_summary"
)

In [0]:
gold_store_summary = silver_df.groupBy(
    "store_id"
).agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue"),
    sum("profit").alias("total_profit")
)

display(gold_store_summary)

store_id,total_quantity,total_revenue,total_profit
S101,8,255000.0,45000.0
S102,20,172500.0,32500.0
S104,10,80000.0,19500.0
S103,30,42700.0,13400.0
S105,12,82500.0,18300.0


In [0]:
gold_store_summary.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.retail_sales_dashboard.gold_store_summary"
)

In [0]:
from pyspark.sql.functions import desc

top_products = gold_product_summary.orderBy(
    desc("total_quantity")
)

display(top_products)

product_id,product_name,total_quantity,total_revenue,total_profit
P108,Mouse,19,15200.0,5700.0
P107,Keyboard,15,22500.0,7500.0
P103,Headphones,11,27500.0,7700.0
P110,Webcam,9,22500.0,6300.0
P104,Smartwatch,7,35000.0,10500.0
P102,Smartphone,5,150000.0,25000.0
P106,Printer,5,60000.0,15000.0
P101,Laptop,3,195000.0,30000.0
P109,Monitor,3,45000.0,9000.0
P105,Tablet,3,60000.0,12000.0


In [0]:
top_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.retail_sales_dashboard.gold_top_products"
)

In [0]:
underperforming_products = gold_product_summary.filter(
    gold_product_summary.total_quantity < 5
)

display(underperforming_products)

product_id,product_name,total_quantity,total_revenue,total_profit
P101,Laptop,3,195000.0,30000.0
P109,Monitor,3,45000.0,9000.0
P105,Tablet,3,60000.0,12000.0


In [0]:
underperforming_products.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
"sathya_databricks123.retail_sales_dashboard.gold_underperforming_products"
)